# `chefai.extractor` — User Guide

Parse Italian cookbook markdown files and export each recipe to its own `.md` or `.json` file, or combine all section files into a single corpus JSON.

```
chefai/extractor.py
│
├── Parsing
│   └── parse_preparazioni_md(filepath)              → List[Dict]
│
├── Utilities
│   └── sanitize_filename(title, max_length=80)      → str
│
└── Export
    ├── export_recipes_to_markdown(recipes, ...)     → None  (one .md per recipe)
    ├── export_recipes_to_json(recipes, ...)         → None  (one .json per recipe)
    └── export_corpus_to_json(recipes, output_path)  → None  (single combined array)
```

| Function | Needs API key | Reads disk | Writes disk |
|---|---|---|---|
| `parse_preparazioni_md` | no | yes | no |
| `sanitize_filename` | no | no | no |
| `export_recipes_to_markdown` | no | no | yes |
| `export_recipes_to_json` | no | no | yes |
| `export_corpus_to_json` | no | no | yes (atomic) |

**All sections run with no API key.**

## Sections
1. [Setup](#1--setup)
2. [Sample data](#2--sample-data)
3. [`parse_preparazioni_md`](#3--parse_preparazioni_md)
4. [`sanitize_filename`](#4--sanitize_filename)
5. [`export_recipes_to_markdown`](#5--export_recipes_to_markdown)
6. [End-to-end pipeline](#6--end-to-end-pipeline)
7. [Error handling reference](#7--error-handling-reference)

## 1 — Setup

In [ ]:
import logging
import tempfile
from pathlib import Path

from chefai.extractor import (
    export_corpus_to_json,
    export_recipes_to_json,
    export_recipes_to_markdown,
    parse_preparazioni_md,
    sanitize_filename,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
    force=True,
)

print("chefai.extractor loaded OK")

## 2 — Sample data

Two sources of recipe data are used throughout this guide:

- **`REAL_FILE`** — the actual `data/processed/preparazioni-di-base.md` shipped with the project (18 recipes across pages 15–22).
- **`MINI_MD`** — a self-contained synthetic markdown string used in cells that must not touch the filesystem or that demonstrate edge cases. It has the same format as the real file but fits in 15 lines.

In [ ]:
# Real file (used in sections 3 and 6)
REAL_FILE = Path("data/processed/preparazioni-di-base.md")
print(f"Real file exists : {REAL_FILE.exists()}")

# Synthetic two-recipe markdown (used in sections 3, 4, 5 for reproducibility)
MINI_MD = """\
---
titolo: PREPARAZIONI DI BASE
pagine: 1–2
---

```text
--- pagina 1 ---
PREPARAZIONI DI BASE
--- pagina 2 ---
BESCIAMELLA
INGREDIENTI
40 G DI BURRO CHIARIFICATO,
50 G DI FARINA,
5 DL DI LATTE,
SALE
Fate fondere il burro e unite la farina. Mescolate e cuocete per 5 minuti.
Unite il latte freddo e fate sobbollire per 10 minuti.

--- pagina 3 ---
BRODO DI CARNE
INGREDIENTI
600 G DI PUNTA DI PETTO DI MANZO,
1 CIPOLLA,
SALE
Coprite la carne con abbondante acqua e fate sobbollire per 4 ore.
Filtrate e salate.
```
"""

print(f"Mini-MD lines      : {len(MINI_MD.splitlines())}")

## 3 — `parse_preparazioni_md`

```python
parse_preparazioni_md(filepath: str | Path) -> List[Dict[str, Any]]
```

Reads a single markdown file and returns a list of recipe dicts.

Works with all 11 section files in `data/processed/`. Both the bare
`INGREDIENTI` keyword and the `INGREDIENTI PER N PERSONE` variant are
recognised. Metadata lines (`PREPARAZIONE:`, `COTTURA:`, `DIFFICOLTÀ:`)
are extracted into structured fields.

### How the parser works

The function tokenises the embedded ` ```text ``` ` block line-by-line through a **7-step state machine**:

| Priority | Condition | Action |
|---|---|---|
| 1 | `--- pagina N ---` | Update `current_page`, no content emitted |
| 2 | Blank line | Flush pending instruction lines as one step |
| 3 | `^INGREDIENTI\b` (bare or `PER N PERSONE`) | Enter ingredient mode; extract `servings` |
| 3.5 | `^(PREPARAZIONE\|COTTURA\|DIFFICOLTÀ):` | Extract `prep_time` / `cook_time` / `difficulty`; skip |
| 4 | **`in_ingredients=True`** (checked before title!) | ALL-CAPS → ingredient (split on `, <qty>`); mixed-case → start instructions |
| 5 | `_is_recipe_title` passes | Close previous recipe, open new one |
| 6 | Everything else | Append to pending instruction buffer |

> **Key invariant — step 4 before step 5:** bare ALL-CAPS words such as `SALE`
> (salt without a unit) appear as the final ingredient in many recipes. If step 5
> ran first they would be misidentified as new recipe titles and eat the recipe's
> instructions. The ordering prevents that bug.
>
> **Key invariant — step 3.5 before step 4:** metadata lines like `DIFFICOLTÀ: *`
> are ALL-CAPS with no digits or units. Without step 3.5 (and the `:` guard in
> `_is_recipe_title`) they would open spurious recipes.

### Returned dict schema

```python
{
    "title":        str,          # Title-cased, e.g. "Brodo Di Carne"
    "ingredients":  List[str],    # one entry per ingredient; multi-ingredient lines are split
    "instructions": List[str],    # one entry per blank-line-separated paragraph
    "pages":        List[int],    # all page numbers the recipe spans
    "raw_text":     str,          # original lines, useful for debugging
    # optional — present only when found in the source:
    "prep_time":    str,          # e.g. "20 MINUTI" or "30 MINUTI + RIPOSO"
    "cook_time":    str,          # e.g. "NO" or "15 MINUTI"
    "difficulty":   str,          # e.g. "*" or "**"
    "servings":     str,          # e.g. "4"
}
```

In [ ]:
# --- Parse from a temporary file built from MINI_MD ---
import tempfile, textwrap

with tempfile.NamedTemporaryFile(
    mode="w", suffix=".md", encoding="utf-8", delete=False
) as tmp:
    tmp.write(MINI_MD)
    tmp_path = Path(tmp.name)

recipes = parse_preparazioni_md(tmp_path)
tmp_path.unlink()  # clean up

print(f"Recipes found : {len(recipes)}\n")
for r in recipes:
    print(f"  Title        : {r['title']}")
    print(f"  Pages        : {r['pages']}")
    print(f"  Ingredients  : {r['ingredients']}")
    print(f"  Instructions : {len(r['instructions'])} step(s)")
    print(f"  Step 1       : {textwrap.shorten(r['instructions'][0], 70)}")
    print()

### 3a — Inspecting multi-paragraph instructions

A blank line in the source text separates instruction paragraphs. Each paragraph becomes one list entry in `instructions`.

In [ ]:
besciamella = recipes[0]
print(f"Title              : {besciamella['title']}")
print(f"Instruction steps  : {len(besciamella['instructions'])}")
for i, step in enumerate(besciamella["instructions"], 1):
    print(f"  Step {i}: {step}")

### 3b — Multi-page tracking

When a recipe straddles a `--- pagina N ---` marker, **both** page numbers are accumulated into `pages`. This makes it easy to trace a recipe back to its source pages in the physical book.

In [ ]:
# The real file has several multi-page recipes — demonstrate with it if available
if REAL_FILE.exists():
    all_recipes = parse_preparazioni_md(REAL_FILE)
    multi_page = [r for r in all_recipes if len(r["pages"]) > 1]
    print(f"Total recipes parsed   : {len(all_recipes)}")
    print(f"Multi-page recipes     : {len(multi_page)}\n")
    for r in multi_page:
        print(f"  {r['title']:30s}  pages = {r['pages']}")
else:
    print("Real file not found — skipping (run from project root)")

## 4 — `sanitize_filename`

```python
sanitize_filename(title: str, max_length: int = 80) -> str
```

Converts an arbitrary recipe title into a filesystem-safe slug suitable for use as a `.md` filename.

**Transformation pipeline:**

| Step | Operation | Example |
|---|---|---|
| 1 | NFKD normalise + drop non-ASCII | `"Crêpe"` → `"Crepe"` |
| 2 | Lowercase, strip non-alphanumeric | `"Ragù (base)"` → `"ragu base"` |
| 3 | Collapse whitespace → hyphens | `"ragu base"` → `"ragu-base"` |
| 4 | Collapse repeated hyphens | `"a--b"` → `"a-b"` |
| 5 | Truncate to `max_length`, strip trailing `-` | `"a"*100` → `"a"*80` |
| 6 | Empty result → sentinel | `"!!!"` → `"ricetta-senza-titolo"` |

In [ ]:
cases = [
    ("Besciamella",        "plain ASCII"),
    ("Crepe",              "accented char (normalized before call)"),
    ("Ragu",               "grave accent (normalized before call)"),
    ("Pasta All'Uovo",     "apostrophe stripped"),
    ("Court-Bouillon",     "hyphen preserved"),
    ("Ragu (base)",        "parentheses stripped"),
    ("!!!",                "symbols only -> sentinel"),
    ("",                   "empty -> sentinel"),
    ("a" * 100,            "truncation at 80 chars"),
]

print(f"{'Input':<30}  {'Result':<35}  Note")
print("-" * 80)
for title, note in cases:
    slug = sanitize_filename(title)
    print(f"{repr(title):<30}  {slug:<35}  {note}")

## 5 — `export_recipes_to_markdown`

```python
export_recipes_to_markdown(
    recipes:          List[Dict[str, Any]],
    output_dir:       str | Path = "recipes_md",
    include_raw_text: bool = True,
    include_pages:    bool = True,
) -> None
```

Writes one `.md` file per recipe into `output_dir` (created if necessary).

### Output file format

Each file contains:
1. **YAML front-matter** — `title`, optional `prep_time/cook_time/difficulty/servings`, `pages`, `confidence` (only if < 0.9)
2. `# Title` heading
3. `### Informazioni` block (only when timing/difficulty fields are present)
4. `## Ingredienti` bullet list
5. `## Procedimento` numbered steps
6. `## Testo originale` fenced code block (controlled by `include_raw_text`)

### Duplicate-filename deduplication

If two recipes produce the same slug (e.g. both called `"Same"`), the second gets `_1` appended, the third `_2`, etc. The counter resets per output directory, not globally.

### Optional fields in the recipe dict

Fields beyond the five required ones are rendered when present:

| Key | Rendered as | Example value |
|---|---|---|
| `prep_time` | `- **Prep:** …` | `"10 min"` |
| `cook_time` | `- **Cottura:** …` | `"1 ora"` |
| `difficulty` | `- **Difficoltà:** …` | `"Facile"` |
| `servings` | (front-matter only) | `"4"` |
| `confidence` | front-matter, only if < 0.9 | `0.72` |

In [ ]:
# Export the two mini recipes to a temp dir and inspect one output file
with tempfile.TemporaryDirectory() as out_dir:
    export_recipes_to_markdown(recipes, output_dir=out_dir, include_raw_text=False)

    files = sorted(Path(out_dir).glob("*.md"))
    print(f"Files written: {[f.name for f in files]}\n")

    print("--- besciamella.md ---")
    print(files[0].read_text(encoding="utf-8"))

### 5a — Using optional metadata fields

Supply `prep_time`, `cook_time`, `difficulty`, or `confidence` in the recipe dict to populate the `### Informazioni` block and front-matter.

In [ ]:
enriched = [
    {
        "title": "Besciamella",
        "ingredients": ["40 G DI BURRO", "50 G DI FARINA", "5 DL DI LATTE"],
        "instructions": ["Fate fondere il burro e unite la farina. Cuocete 5 minuti."],
        "pages": [16],
        "prep_time": "5 min",
        "cook_time": "15 min",
        "difficulty": "Facile",
        "confidence": 0.85,   # below 0.9 → appears in front-matter
    }
]

with tempfile.TemporaryDirectory() as out_dir:
    export_recipes_to_markdown(enriched, output_dir=out_dir, include_raw_text=False)
    print((Path(out_dir) / "besciamella.md").read_text(encoding="utf-8"))

### 5b — Duplicate filename deduplication

When two recipes normalise to the same slug, the exporter appends `_1`, `_2`, … rather than overwriting the first file.

In [ ]:
dupes = [
    {"title": "Brodo", "ingredients": ["ACQUA"], "instructions": ["Portate a ebollizione."]},
    {"title": "Brodo", "ingredients": ["ACQUA", "VERDURE"], "instructions": ["Fate sobbollire."]},
    {"title": "Brodo", "ingredients": ["ACQUA", "CARNE"], "instructions": ["Cuocete per ore."]},
]

with tempfile.TemporaryDirectory() as out_dir:
    export_recipes_to_markdown(dupes, output_dir=out_dir, include_raw_text=False)
    files = sorted(f.name for f in Path(out_dir).glob("*.md"))
    print(f"Files created: {files}")
    # Confirm each file has distinct ingredient count
    for fname in files:
        content = (Path(out_dir) / fname).read_text(encoding="utf-8")
        ing_count = content.count("- ")
        print(f"  {fname}: {ing_count} ingredient(s)")

## 6 — End-to-end pipeline

The canonical usage pattern: **parse → inspect → export**.

```
parse_preparazioni_md(file)
        │
        ▼
  List[Dict]   ←── optionally enrich each dict with prep_time, difficulty, …
        │
        ▼
export_recipes_to_markdown(recipes, output_dir)
        │
        ▼
  recipes_md/
    ├── besciamella.md
    ├── brodo-di-carne.md
    └── …
```

In [ ]:
def parse_and_export(source_file: Path, output_dir: Path) -> list:
    """
    Full pipeline: parse *source_file* and write one .md per recipe into *output_dir*.

    Returns the list of parsed recipe dicts so callers can inspect or
    post-process them without re-reading the disk output.
    """
    recipes = parse_preparazioni_md(source_file)
    export_recipes_to_markdown(recipes, output_dir=output_dir, include_raw_text=False)
    return recipes


if REAL_FILE.exists():
    with tempfile.TemporaryDirectory() as out:
        out_path = Path(out)
        all_recipes = parse_and_export(REAL_FILE, out_path)

        files = sorted(out_path.glob("*.md"))
        print(f"Parsed   : {len(all_recipes)} recipes")
        print(f"Exported : {len(files)} files")
        print(f"\nFirst 5 files:")
        for f in files[:5]:
            print(f"  {f.name}")

        # Spot-check: confirm Burro Chiarificato spans two pages
        burro = next(r for r in all_recipes if "Burro Chiarificato" in r["title"])
        print(f"\nBurro Chiarificato pages: {burro['pages']}")
else:
    print("Real file not available — run from project root")

## 7 — Error handling reference

### Raises summary

| Function | Condition | Exception |
|---|---|---|
| `parse_preparazioni_md` | File path does not exist | `FileNotFoundError` |
| `parse_preparazioni_md` | No ` ```text ``` ` block in file | `ValueError` |
| `parse_preparazioni_md` | Recipe with 0 ingredients or 0 instructions | `WARNING` log (no exception) |
| `export_recipes_to_markdown` | `recipes` is not a `list` | `ValueError` |
| `export_recipes_to_markdown` | `output_dir` path is a regular file | `OSError` |
| `export_recipes_to_markdown` | Empty `recipes` list | `WARNING` log, no-op |
| `export_recipes_to_json` | `recipes` is not a `list` | `ValueError` |
| `export_recipes_to_json` | `output_dir` path is a regular file | `OSError` |
| `export_recipes_to_json` | Empty `recipes` list | `WARNING` log, no-op |
| `export_corpus_to_json` | `recipes` is not a `list` | `ValueError` |
| `export_corpus_to_json` | `output_path` is an existing directory | `OSError` |
| `export_corpus_to_json` | Empty `recipes` list | `WARNING` log, no-op |

### Debugging a parsing anomaly

If a recipe shows up with no ingredients or no instructions in the WARNING logs:
1. Open the source `.md` and find the recipe name.
2. Check whether a bare ALL-CAPS word (e.g. `OLIO`, `PEPE`) appears on its own line — it will be treated as an ingredient, not a title.
3. Check whether the `INGREDIENTI` (or `INGREDIENTI PER N PERSONE`) header is present and correctly spelt.
4. Check for unexpected metadata lines between the title and `INGREDIENTI` — these must start with `PREPARAZIONE:`, `COTTURA:`, or `DIFFICOLTÀ:` to be skipped correctly.
5. Run `parse_preparazioni_md` in isolation and inspect `r["raw_text"]` for the suspicious recipe to see what lines were captured.

In [ ]:
# --- Demonstrate each guard ---

# 1. FileNotFoundError
try:
    parse_preparazioni_md("ghost.md")
except FileNotFoundError as e:
    print(f"FileNotFoundError : {e}")

# 2. ValueError — no ```text block
with tempfile.NamedTemporaryFile(
    mode="w", suffix=".md", encoding="utf-8", delete=False
) as f:
    f.write("---\ntitolo: EMPTY\n---\nNo code block here.\n")
    bad_path = Path(f.name)

try:
    parse_preparazioni_md(bad_path)
except ValueError as e:
    print(f"ValueError        : {e}")
finally:
    bad_path.unlink()

# 3. export_recipes_to_markdown — wrong type
try:
    export_recipes_to_markdown("not a list")
except ValueError as e:
    print(f"ValueError        : {e}")

# 4. export_recipes_to_markdown — output_dir is a file
with tempfile.NamedTemporaryFile(suffix=".txt", delete=False) as f:
    collision_path = Path(f.name)

try:
    export_recipes_to_markdown(
        [{"title": "X", "ingredients": [], "instructions": []}],
        output_dir=collision_path,
    )
except OSError as e:
    print(f"OSError           : {e}")
finally:
    collision_path.unlink()

# 5. export_corpus_to_json — output_path is a directory
import tempfile as _tf
with _tf.TemporaryDirectory() as dir_path:
    try:
        export_corpus_to_json(
            [{"title": "X", "ingredients": [], "instructions": []}],
            output_path=Path(dir_path),
        )
    except OSError as e:
        print(f"OSError           : {e}")